# Notebook 12 — Inside the Black Box
**AI Models and Agents for Business · Week 6 · Session 12**

Karpathy's talk explained what an LLM does. This notebook lets you *watch* it happen — one token at a time, with the actual probabilities exposed.

**No API key needed.** We run a small model (`distilgpt2`, 82M parameters) locally in Colab so you can see its internals. The frontier models you use in other notebooks share the same core loop (predict a probability for every possible next token, pick one, repeat), but they are vastly bigger, differ in architecture details, and go through much more training, including the fine-tuning and human-feedback steps you'll see in Session 13. This toy model shows the loop, not everything a frontier model is.

| Part | You'll see |
|---|---|
| A | Text becoming tokens — and why tokens, not words, drive your bill |
| B | The model's actual next-token probability distribution |
| C | What temperature really does to that distribution |
| D | The context window failing, live |

⚠️ **Cell 1 downloads a model (~350 MB).** It takes 1–2 minutes on classroom wifi. Run it as soon as you sit down.

## Cell 1 — Setup

This cell always runs as-is. Be patient on the first run.

In [ ]:
!pip install transformers torch --quiet

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

print(f"Loaded {MODEL_NAME}")
print(f"Parameters      : {model.num_parameters():,}")
print(f"Vocabulary size : {tokenizer.vocab_size:,} tokens")
print(f"Context window  : {model.config.n_positions} tokens")

---
## Part A: Text is not words. Text is tokens.

A model never sees letters or words. It sees integers, each standing for a chunk of text called a **token**. Roughly: 1 token ≈ 4 characters ≈ 0.75 words in English — but the details are surprising and they cost you money.

In [ ]:
def show_tokens(text):
    ids = tokenizer.encode(text)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f'Text  : "{text}"')
    print(f"Chars : {len(text)}   Tokens: {len(ids)}")
    print("Split : " + " | ".join(repr(p) for p in pieces))
    print()

show_tokens("The quarterly earnings exceeded expectations.")
show_tokens("Klarna deployed an AI assistant.")
show_tokens("EBITDA margin compression")
show_tokens("antidisestablishmentarianism")
show_tokens("$1,234,567.89")
show_tokens("Straße Tokyo 東京")

### 💬 Discussion — Part A
- Compare `"EBITDA margin compression"` to a plain-English phrase of similar length. Which uses more tokens per word, and why? *(Rare and technical words get shattered into fragments.)*
- Look at the number and the non-English examples. What does that imply about the cost of running the same AI product in English versus Japanese, or on financial jargon versus everyday prose?
- **Business consequence:** you're pricing an AI feature for a global customer base. What did you just learn that belongs in your cost model?

## Cell 3 — ✏️ CHANGE ME: tokenize your own text

Paste in something from your project domain — a product description, a legal clause, a support ticket.

In [ ]:
# ✏️ CHANGE ME
MY_TEXT = "Our Q3 EBITDA margin compressed 240bps due to FX headwinds in APAC."

ids = tokenizer.encode(MY_TEXT)
print(f"Characters : {len(MY_TEXT)}")
print(f"Words      : {len(MY_TEXT.split())}")
print(f"Tokens     : {len(ids)}")
print(f"Tokens per word: {len(ids)/len(MY_TEXT.split()):.2f}")
print()
# What a million calls of this would cost at a representative API price
PRICE_PER_MILLION_INPUT_TOKENS = 0.075    # ✏️ CHANGE ME — check your provider's current rate (gpt-oss-20b on Groq, Oct 2026)
cost_one = len(ids) / 1_000_000 * PRICE_PER_MILLION_INPUT_TOKENS
print(f"Input cost per call     : ${cost_one:.8f}")
print(f"Input cost per 1M calls : ${cost_one * 1_000_000:,.2f}")
print()
print("Token-by-token:")
print(" | ".join(repr(tokenizer.decode([i])) for i in ids))

---
## Part B: The model is a next-token probability machine

This is the whole mechanism. Given some text, the model outputs a probability for **every one of its 50,257 tokens** being the next one. That's it. Everything else — chat, reasoning, code generation — is this operation repeated.

Let's look at the actual distribution.

In [ ]:
def next_token_probs(prompt, top_k=10):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]   # scores for the NEXT token
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, top_k)
    return [(tokenizer.decode([i]), float(p)) for i, p in zip(top.indices, top.values)]

PROMPT = "The company reported a significant increase in quarterly"

print(f'Prompt: "{PROMPT}"')
print(f"\nThe model considered all {tokenizer.vocab_size:,} tokens. Top 10:\n")
results = next_token_probs(PROMPT)
for token, prob in results:
    bar = "█" * int(prob * 120)
    print(f"{repr(token):>14}  {prob:6.2%}  {bar}")

total_top10 = sum(p for _, p in results)
print(f"\nTop 10 tokens hold {total_top10:.1%} of the probability mass.")
print(f"The remaining {tokenizer.vocab_size - 10:,} tokens share {1-total_top10:.1%}.")

## Cell 5 — ✏️ CHANGE ME: probe the model's confidence

Try prompts where the next token is nearly certain versus genuinely open-ended, and compare the shape of the distribution:

- `"The capital of France is"` → should be very peaked
- `"My favorite thing about weekends is"` → should be flat and uncertain
- `"2 + 2 ="` → try it; small models are bad at arithmetic and you'll see why

In [ ]:
# ✏️ CHANGE ME
for prompt in ["The capital of France is",
               "My favorite thing about weekends is",
               "2 + 2 ="]:
    results = next_token_probs(prompt, top_k=5)
    top_prob = results[0][1]
    confidence = "VERY CONFIDENT" if top_prob > 0.5 else ("uncertain" if top_prob < 0.15 else "moderate")
    print(f'"{prompt}"  ->  {confidence} (top token {top_prob:.1%})')
    for t, p in results:
        print(f"      {repr(t):>12} {p:6.2%}")
    print()

### 💬 Discussion — Part B
- On the open-ended prompt, no single token gets much probability. The model still has to pick one. Where does that choice come from? *(Sampling — Part C.)*
- Notice the model has no concept of "I don't know." It always produces a distribution over tokens. Connect this to the hallucination probe you'll run in Session 21: if the model must always output *something*, what happens when it has no real information?
- A frontier model runs this same next-token operation, with far more parameters and much more training. Does more scale eliminate the "always outputs a distribution" property, or just sharpen it?

---
## Part C: What temperature actually does

Every model call you have made in this course sets a `temperature`. Here's what it actually does: temperature rescales the probability distribution *before* a token is sampled.

- **Low temperature** → sharpens toward the most likely token (more deterministic)
- **High temperature** → flattens the distribution (more random)

In [ ]:
def probs_at_temperature(prompt, temperature, top_k=8):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
    scaled = logits / temperature
    probs = torch.softmax(scaled, dim=-1)
    top = torch.topk(probs, top_k)
    return [(tokenizer.decode([i]), float(p)) for i, p in zip(top.indices, top.values)]

PROMPT = "The customer service team decided to"
temps = [0.2, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, T in zip(axes, temps):
    res = probs_at_temperature(PROMPT, T)
    labels = [repr(t) for t, _ in res][::-1]
    vals = [p for _, p in res][::-1]
    ax.barh(labels, vals)
    ax.set_title(f"temperature = {T}\ntop token: {res[0][1]:.1%}")
    ax.set_xlabel("probability")
axes[0].set_ylabel("candidate next token")
plt.suptitle(f'Same prompt, three temperatures: "{PROMPT}"', y=1.03)
plt.tight_layout(); plt.show()

for T in temps:
    res = probs_at_temperature(PROMPT, T)
    spread = res[0][1] - res[-1][1]
    print(f"T={T}: top token {res[0][1]:.1%}, 8th token {res[-1][1]:.2%}, spread {spread:.1%}")

### 💬 Discussion — Part C
- At `T=0.2`, how often would you expect the model to produce the same output twice for the same prompt? At `T=2.0`?
- You're building two features: (1) extracting a dollar amount from an invoice, (2) generating three alternative marketing taglines. What temperature for each, and why?
- Temperature does **not** add knowledge — it only reweights what's already there. Why does that mean you can't fix a wrong answer by turning temperature down?

---
## Part D: The context window, failing live

`distilgpt2` has a hard limit of 1,024 tokens. Anything beyond that simply does not exist for the model. Modern models have much bigger windows, but the property is the same: **a hard edge, past which information is invisible.**

Below we bury a specific fact at the start of a long document and then ask about it — first with the full document, then with the beginning truncated away.

In [ ]:
SECRET = "The acquisition price was 4.7 billion dollars."
FILLER = ("The committee reviewed the standard operating procedures in detail. "
          "Various stakeholders provided input on the quarterly process. ")

# Build a long document with the key fact at the very beginning
document = SECRET + " " + FILLER * 60
question = " In summary, the acquisition price was"

full_ids = tokenizer.encode(document + question)
print(f"Full document + question : {len(full_ids)} tokens")
print(f"Model's hard limit       : {model.config.n_positions} tokens")
print(f"Over the limit by        : {max(0, len(full_ids) - model.config.n_positions)} tokens")
print()

def answer_from(text, label):
    ids = tokenizer.encode(text)
    # Keep only the LAST n_positions tokens — exactly what happens when you overflow
    ids = ids[-model.config.n_positions:]
    kept = tokenizer.decode(ids)
    has_fact = "4.7 billion" in kept
    inputs = torch.tensor([ids])
    with torch.no_grad():
        logits = model(inputs).logits[0, -1, :]
    top = torch.topk(torch.softmax(logits, dim=-1), 5)
    guesses = [tokenizer.decode([i]) for i in top.indices]
    print(f"{label}")
    print(f"  tokens kept          : {len(ids)}")
    print(f"  key fact still there : {'YES' if has_fact else 'NO — it fell outside the window'}")
    print(f"  model's top guesses  : {guesses}")
    print()

answer_from(SECRET + question, "A) Short context — fact is right there:")
answer_from(document + question, "B) Long context — fact overflowed and was cut:")

### 💬 Discussion — Part D
- In case B the fact was in the document you supplied, but *not* in what the model actually received. From the outside, the model looks like it "forgot." What really happened?
- This is why long chat conversations start losing earlier details, and why Notebook 10's "the API resends the whole history every time" matters both for cost *and* for correctness.
- **Bridge to Module 4:** if the information you need doesn't fit in the window, you can't just paste more. You need to *retrieve* only the relevant pieces first. That's RAG — Sessions 17 and 18.
- Frontier models now advertise windows of 200K+ tokens. Does that make this problem go away, or move it? *(Consider cost per call, and whether a model attends equally well to material buried in the middle of a very long input.)*

## ✅ Notebook 12 complete — Reflection questions

These prep you for **Quiz 4** (Session 16).

1. Why is "tokens" the right unit for pricing an LLM rather than words or characters?
2. Explain in one sentence what the model computes at each step.
3. Why can't lowering temperature make a factually wrong answer correct?
4. A colleague says their AI feature "forgot" an instruction from earlier in a long conversation. Give the mechanical explanation.
5. Your product will process German and Japanese customer emails. What did Part A teach you that affects your cost forecast?

---
*Next session: Applied prompt engineering — now that you know what's happening underneath, you'll shape it deliberately.*